# Containerize an AI App

This notebook provides a complete example of containerizing a FastAPI + AI backend application. All Docker commands run in your terminal.

## 1. Project Structure

```
ai-backend/
├── main.py              # FastAPI application
├── requirements.txt     # Python dependencies
├── Dockerfile           # Container build instructions
├── docker-compose.yml   # Multi-service orchestration
├── .dockerignore        # Files to exclude from build
├── .env.example         # Environment variable template
└── .env                 # Your actual secrets (git-ignored)
```

## 2. Complete FastAPI Application

Create `main.py`:

```python
import os
import logging
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()

logger = logging.getLogger(__name__)

app = FastAPI(title="AI Backend", version="1.0.0")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME", "gemini-pro")


class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 100
    temperature: float = 0.7


class GenerateResponse(BaseModel):
    text: str
    model: str
    tokens_used: int


@app.get("/health")
async def health():
    return {"status": "healthy", "model": MODEL_NAME}


@app.get("/ready")
async def ready():
    if not OPENAI_API_KEY:
        raise HTTPException(status_code=503, detail="API key not configured")
    return {"status": "ready"}


@app.post("/generate", response_model=GenerateResponse)
async def generate(request: GenerateRequest):
    logger.info(f"Generate request: model={MODEL_NAME}, prompt_len={len(request.prompt)}")
    
    # Replace with actual LLM call
    # Example with openai:
    # import openai
    # API_KEY = os.getenv("OPENAI_API_KEY")
    # client = openai.OpenAI(api_key=API_KEY)
    # model = openai.OpenAI()
    # response = client.chat.completions.create(request.prompt)
    
    # Placeholder response
    return GenerateResponse(
        text=f"Response to: {request.prompt}",
        model=MODEL_NAME,
        tokens_used=len(request.prompt.split())
    )
```

**Note**: This is a simplified example. In production, you'd add proper LLM integration, error handling, and rate limiting.

## 3. Requirements File

Create `requirements.txt`:

```
fastapi==0.115.0
uvicorn[standard]==0.30.0
pydantic==2.9.0
python-dotenv==1.0.1
openai==0.8.0
```

**Tip**: Pin exact versions for reproducibility. Use `pip freeze > requirements.txt` after testing locally.

## 4. Complete Dockerfile

Create `Dockerfile`:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

# System dependencies (add if needed)
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Install Python dependencies (cached layer)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Non-root user for security
RUN useradd --create-home appuser
USER appuser

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

## 5. Docker Compose (Multi-Service)

For apps with multiple services (e.g., API + Redis cache + worker):

Create `docker-compose.yml`:

```yaml
version: '3.8'

services:
  api:
    build: .
    ports:
      - "8000:8000"
    env_file:
      - .env
    environment:
      - REDIS_URL=redis://redis:6379
    depends_on:
      - redis
    restart: unless-stopped

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data

volumes:
  redis_data:
```

**Commands**:

```bash
# Start all services
docker compose up

# Start in detached mode
docker compose up -d

# Stop all services
docker compose down

# Rebuild after code changes
docker compose up --build

# View logs
docker compose logs -f api
```

## 6. Environment Variables in Docker

### .dockerignore

Create `.dockerignore` to prevent secrets from being baked into the image:

```gitignore
.git
.env
.env.local
.env.*
__pycache__
*.pyc
.venv
venv
.pytest_cache
```

### .env.example (commit this)

```
OPENAI_API_KEY=your-key-here
PINECONE_API_KEY=your-key-here
PINECONE_INDEX=your-index
MODEL_NAME=gemini-pro
REDIS_URL=redis://localhost:6379
```

### Passing Secrets at Runtime

```bash
# Method 1: From .env file
docker run --env-file .env -p 8000:8000 ai-backend

# Method 2: Individual variables
docker run -e OPENAI_API_KEY=xxx -e MODEL_NAME=gemini-pro -p 8000:8000 ai-backend

# Method 3: Docker Compose (uses .env automatically)
docker compose up
```

## 7. Build and Run Locally

### Step-by-Step Terminal Commands

```bash
# 1. Navigate to project directory
cd ai-backend

# 2. Create your .env file (copy from .env.example and fill in real values)
cp .env.example .env
nano .env  # or use your editor

# 3. Build the Docker image
docker build -t ai-backend:latest .

# 4. Check the image was created
docker images | grep ai-backend

# 5. Run the container
docker run -d --name ai-backend -p 8000:8000 --env-file .env ai-backend:latest

# 6. Check if it's running
docker ps

# 7. Test the health endpoint
curl http://localhost:8000/health
# Expected: {"status":"healthy","model":"gemini-pro"}

# 8. Test the generate endpoint
curl -X POST http://localhost:8000/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt": "Hello, AI!", "max_tokens": 50}'

# 9. View logs
docker logs ai-backend

# 10. Stop and remove
docker stop ai-backend
docker rm ai-backend
```

## 8. Troubleshooting Common Issues

| Problem | Cause | Solution |
|---------|-------|----------|
| `ModuleNotFoundError` | Missing dependency in requirements.txt | Add it and rebuild |
| Port already in use | Another process on port 8000 | `lsof -i :8000` to find, `kill <PID>` |
| Container exits immediately | App crashes on startup | `docker logs <container>` to see error |
| Cannot connect to localhost | App binds to 127.0.0.1 | Use `--host 0.0.0.0` in uvicorn |
| Slow build | No layer caching | Order Dockerfile: deps before code |
| Large image size | Using full Python image | Switch to `python:3.11-slim` |
| API key not found | .env not passed to container | Use `--env-file .env` flag |

### Debug Commands

```bash
# Enter a running container
docker exec -it ai-backend /bin/bash

# Check environment variables inside container
docker exec ai-backend env

# Inspect container details
docker inspect ai-backend

# View container resource usage
docker stats ai-backend

# Check image size
docker images ai-backend
```

## Summary

| Step | Command | Purpose |
|------|---------|--------|
| Build | `docker build -t name .` | Create image from Dockerfile |
| Run | `docker run -p 8000:8000 --env-file .env name` | Start container |
| Test | `curl localhost:8000/health` | Verify it works |
| Logs | `docker logs name` | Debug issues |
| Stop | `docker stop name` | Shut down |

**Next**: [03-cloud-run-deploy.ipynb](03-cloud-run-deploy.ipynb) — Deploy to Google Cloud Run.